# 3 Run the method end to end

Six phases: data → warmup → clustering → expert training → combiner fitting → evaluation.

> **Scale warning.** The `fast` config below subsamples CIFAR-10 so it finishes on a laptop, which starves each expert of gradient steps (~200, where ResNet-18 from scratch needs ~10⁴). Every row will land at chance. Use it to verify the pipeline, **not** to compare methods — see notebook 05 for the GPU path.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np, torch, matplotlib.pyplot as plt
from hefl.utils import pick_device, set_seed
set_seed(42); DEVICE = pick_device('auto')
print('device:', DEVICE)


## The architecture in one cell

K cluster backbones, each with its own linear head, a shared bias, and a **combiner** that decides how the per-expert logits are mixed:

```
logits = Σ_k  w_k · head_k(backbone_k(x))  +  bias
```

The combiner is the part the original formulation left untrained. With `w_k = 1/|A|` this reproduces it exactly.


In [ ]:
from hefl.models import ClusterEnsemble, ClientView
from hefl.federated import ensemble_to_client_state

ens = ClusterEnsemble(num_clusters=3, num_classes=10, norm='gn', combiner='gate').eval()
x = torch.randn(2, 3, 32, 32)

# A client trains through ONE expert. Evaluating the ensemble with only that
# expert active must give the identical function - no magnitude correction.
client = ClientView(norm='gn').eval()
client.load_state_dict(ensemble_to_client_state(ens, 1))
with torch.no_grad():
    a, b = ens(x, active=[1]), client(x)
print(f'ensemble(active=[1]) vs ClientView : max |diff| = {(a-b).abs().max():.2e}')
print(f'parameters: {sum(p.numel() for p in ens.parameters())/1e6:.1f}M for K=3')


## Combiners

| name | params | weights | note |
|---|---|---|---|
| `uniform` | 0 | `1/\|A\|` | reproduces the original method |
| `beta` | K | `softmax(β)` | one global scalar per expert |
| `gate` | ~1k | `softmax(g(conf(x)))` | **input-dependent** — the one that matters under feature shift |
| `mlp` | ~5k | — | genuine cross-expert interaction |


In [ ]:
from hefl.models import build_combiner, confidence_features

stack = torch.randn(4, 3, 10)                       # (batch, experts, classes)
mask  = torch.tensor([True, False, True])           # experts 0 and 2 active
for name in ('uniform', 'beta', 'gate'):
    _, aux = build_combiner(name, 3, 10)(stack, mask)
    w = aux['weights']
    print(f'{name:<9} weights[0] = {w[0].detach().numpy().round(3)}  sum = {w[0].sum():.3f}')
print('\nInactive experts get exactly 0 and the rest renormalise -> every subset is magnitude-correct.')


## Run the pipeline


In [ ]:
%%time
!cd .. && python -m hefl.run --config hefl/configs/fast.json --output_dir ./hefl/results/nb_demo


## Read the result


In [ ]:
import json
res = json.load(open('../hefl/results/nb_demo/results.json'))
print(open('../hefl/results/nb_demo/table.md').read())
print('clustering:')
for sig, m in res['clustering'].items():
    print(f"  {sig:<12} ARI rotation {m['ari_rotation']:.3f}   ARI label {m['ari_label']:.3f}")


## Is this result real?

Three checks, in order. If the first fails, nothing below it is worth reading.


In [ ]:
mat = np.array(res['expert_matrix'])       # (K experts, G rotations)
diag_wins = sum(int(mat[:, g].argmax()) == res['cluster_to_rotation'].get(str(k), -1)
                for g, k in enumerate(range(mat.shape[0])))
oracle = res['oracle']['test']['overall']
fedavg = res.get('fedavg', {}).get('test', {}).get('overall', float('nan'))
gate   = res.get('ensemble_gate', {}).get('test', {}).get('overall', float('nan'))
unif   = res.get('ensemble_uniform', {}).get('test', {}).get('overall', float('nan'))

print(f"1. does expert k win on rotation k?      per-expert argmax = {[int(mat[:, g].argmax()) for g in range(mat.shape[1])]}")
print(f"2. is the oracle above FedAvg?           oracle {oracle:.4f}  vs  FedAvg {fedavg:.4f}  -> {'YES' if oracle > fedavg else 'NO - experts undertrained'}")
print(f"3. does the gate beat uniform?           gate {gate:.4f}  vs  uniform {unif:.4f}")
